SimCLR backbone and texture comparison

Compare ResNet18, ResNet18 + texture, ResNet50 + texture, and ResNet50.
Each configuration has a readable/bad gate and a separate age classifier:
0, 1, 2, and 3 or older. Original label 6 denotes bad, never an age.
All four experiments share fish-level train/validation/test splits. SimCLR
sees training fish only. Hyperparameters, stopping epoch and the quality gate
threshold are selected with validation only. Test is evaluated after selection.
Image-only inputs and texture features contain no fish length or weight.

In [ ]:
import gc
import hashlib
import json
import os
import random
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid, ParameterSampler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, balanced_accuracy_score

ROOT_DIR = Path(os.getenv("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.getenv("TROUT_CODE_DIR", str(ROOT_DIR / "trout_code_new")))
INPUT_CSV = Path(os.getenv("TROUT_TEXTURE_MASTER", str(
    CODE_DIR / "feature_outputs/master_with_texture_features_full.csv")))
if not INPUT_CSV.exists() and "TROUT_TEXTURE_MASTER" not in os.environ:
    legacy_input = ROOT_DIR / "code_new/feature_outputs/master_with_texture_features_full.csv"
    if legacy_input.exists():
        INPUT_CSV = legacy_input
OUTPUT_DIR = CODE_DIR / "model_outputs/age4_two_stage_search"
SEED = 100
CLASS_NAMES = ["0", "1", "2", "3 or older"]
LABELS = list(range(4))
QUALITY_NAMES = ["readable", "bad"]
PIPELINE_NAMES = CLASS_NAMES + ["bad"]
SPLIT_LABELS = list(range(5))
TEST_SIZE = 0.20
VAL_SIZE_OF_REMAINDER = 0.20
# This is budgeted search, not an exhaustive search or a guaranteed optimum.
SEARCH_MODE = "random"  # "grid" evaluates every combination; much more expensive.
N_PRETRAIN_TRIALS = 2  # Per backbone; shared by image-only and fusion variants.
N_CLASSIFIER_TRIALS = 3  # Per pretrained backbone candidate and task.
SIMCLR_SPACE = {"lr": [1e-4, 3e-4], "temperature": [0.1, 0.2, 0.5],
                "batch_size": [64, 128], "epochs": [10, 20]}
CLASSIFIER_SPACE = {"lr": [3e-5, 1e-4, 3e-4], "batch_size": [32, 64]}
CLASSIFIER_MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 1e-4
EVAL_BATCH_SIZE = 64
THRESHOLD_GRID = np.linspace(0.0, 1.0, 41).tolist()
THRESHOLD_OBJECTIVE = "pipeline_macro_f1"  # Or "quality_balanced_accuracy".
MAX_READABLE_REJECTION = None  # Optional validation constraint, e.g. 0.05.
NUM_WORKERS = 0  # Safe default for HPC Jupyter multiprocessing.
USE_AMP = False  # Optional CUDA speed/memory optimization.
RUN_TRAINING = False  # Set True after checking the data/split audit.
EVALUATE_TEST = False  # Enable only when search and threshold selection are complete.
RESUME_SEARCH = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and device.type == "cuda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("device:", device, "input:", INPUT_CSV, "output:", OUTPUT_DIR)
print("torch:", torch.__version__, "AMP:", AMP_ENABLED)

Search protocol

Default: 2 SimCLR candidates per backbone, then 3 classifier candidates per
pretrained candidate for each of four configurations and two tasks (48
classifier trials). Random search includes the previous baseline when it is
in the search space. Increase the budgets to cover more combinations. Grid
mode is exhaustive only within the specified finite spaces.

Search learning rate, temperature, SimCLR batch size/epoch budget, and classifier
learning rate/batch size. Classifier training runs for up to 30 epochs with
validation macro-F1 early stopping and best-epoch checkpoint selection. SimCLR
loss is logged, but candidates are compared using downstream validation
performance, not losses across different temperatures/batch sizes.

Keep fish splits fixed. NUM_WORKERS and AMP are execution settings, not
predictive hyperparameters; retain the safe worker setting unless benchmarking
the assigned CPUs separately. No test scoring occurs during search. Repeated
use of validation can still overfit it; larger searches warrant grouped nested
validation or repeated grouped experiments. Do not pick a new split to improve
test performance.

Input and cohort audit

Run full texture extraction first. Set TROUT_TEXTURE_MASTER to the actual CSV
if prior outputs are elsewhere. The known code_new full CSV is an automatic
fallback; sample CSVs are never used. Compute targets from original label.
Numeric label 6 follows the existing bad convention; confirm ambiguous GT
entries with the experts before interpreting it as a biological age.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Set INPUT_CSV to the full texture master: {INPUT_CSV}")
master = pd.read_csv(INPUT_CSV)
required = {"scale_id", "fish_key", "path", "label"}
if required - set(master):
    raise ValueError(f"Missing columns: {sorted(required - set(master))}")
if master["scale_id"].isna().any() or master["scale_id"].duplicated().any():
    raise ValueError("Master scale_id must be non-null and unique; audit Excel joins.")
numeric_label = pd.to_numeric(master["label"], errors="coerce")
invalid = master["label"].notna() & ~numeric_label.isin(range(7))
if invalid.any():
    raise ValueError("Expected integer labels 0..6 in the master table.")
texture_prefixes = ("intensity_", "sobel_", "canny_", "edge_component_",
                    "glcm_", "lbp_", "gabor_", "hog_")
texture_cols = [c for c in master if c.startswith(texture_prefixes)]
if not texture_cols:
    raise ValueError("No image-derived texture columns found; run full extraction.")
cohort = master.loc[numeric_label.notna()].copy()
cohort["target"] = numeric_label.loc[cohort.index].astype(int).clip(upper=3)
cohort.loc[numeric_label.loc[cohort.index].eq(6), "target"] = 4
if cohort["fish_key"].isna().any():
    raise ValueError("Labeled scales need fish_key for leakage-free splitting.")
cohort["fish_key"] = cohort["fish_key"].astype(str).str.strip().str.lower()
texture_values = cohort[texture_cols].apply(pd.to_numeric, errors="raise")
texture_values = texture_values.replace([np.inf, -np.inf], np.nan)
available = texture_values.notna().any(axis=1)
path_exists = cohort["path"].map(lambda p: Path(str(p)).is_file())
print("Excluded unlabeled rows:", len(master) - len(cohort))
print("Labeled rows missing image:", int((~path_exists).sum()))
print("Labeled rows missing all texture features:", int((~available).sum()))
cohort[texture_cols] = texture_values
# All models use the same available image/texture cohort for a paired comparison.
cohort = cohort.loc[path_exists & available].sort_values("scale_id").reset_index(drop=True)
if cohort["fish_key"].eq("").any():
    raise ValueError("Empty fish_key detected.")
display(cohort.groupby("target").agg(scales=("scale_id", "size"),
                                    fish=("fish_key", "nunique")))
readable_gt = cohort.loc[cohort["target"].lt(4)]
print("Fish with conflicting readable age GT:",
      int((readable_gt.groupby("fish_key")["target"].nunique() > 1).sum()))

Fixed fish-level splits and preprocessing

Reuse the saved split manifest only for an identical cohort, labels and texture
table. Validation is 20% of the remaining 80%; test is 20% of fish. Seed search
only checks class presence, never model performance. Conflicting GT remains
scale-level; all scales belonging to a fish stay together.

In [ ]:
fingerprint_cols = ["scale_id", "fish_key", "path", "target"] + texture_cols
digest = hashlib.sha256(pd.util.hash_pandas_object(
    cohort[fingerprint_cols], index=False).values.tobytes()).hexdigest()
split_path = OUTPUT_DIR / "split_manifest.csv"
split_meta_path = OUTPUT_DIR / "split_metadata.json"
split_config = {"seed": SEED, "test_size": TEST_SIZE,
                "val_size_of_remainder": VAL_SIZE_OF_REMAINDER,
                "cohort_sha256": digest, "texture_cols": texture_cols}

def make_split(df):
    for seed in range(SEED, SEED + 500):
        outer = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        rem_idx, test_idx = next(outer.split(df, groups=df["fish_key"]))
        remainder = df.iloc[rem_idx]
        inner = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_OF_REMAINDER,
                                  random_state=seed + 1000)
        train_sub, val_sub = next(inner.split(remainder, groups=remainder["fish_key"]))
        indices = {"train": rem_idx[train_sub], "validation": rem_idx[val_sub],
                   "test": test_idx}
        if all(set(df.iloc[idx]["target"]) == set(SPLIT_LABELS) for idx in indices.values()):
            manifest = df[["scale_id", "fish_key", "target"]].copy()
            manifest["split"] = ""
            for name, idx in indices.items():
                manifest.loc[idx, "split"] = name
            return manifest, seed
    raise ValueError("Cannot put all four ages and bad in each fish split. Inspect GT fish counts.")

if split_path.exists() or split_meta_path.exists():
    if not (split_path.exists() and split_meta_path.exists()):
        raise ValueError("Incomplete saved split; choose a new OUTPUT_DIR.")
    previous = json.loads(split_meta_path.read_text())
    if any(previous.get(k) != v for k, v in split_config.items()):
        raise ValueError("Cohort/config changed. Choose a new OUTPUT_DIR to preserve the old run.")
    manifest = pd.read_csv(split_path)
else:
    manifest, selected_seed = make_split(cohort)
    manifest.to_csv(split_path, index=False)
    split_meta_path.write_text(json.dumps({**split_config, "selected_seed": selected_seed}, indent=2))
if manifest["scale_id"].isna().any() or manifest["scale_id"].duplicated().any():
    raise ValueError("Split manifest needs unique, non-null scale_id values.")
# Preserve the source labeled/unlabeled split; assign model partitions separately.
cohort["model_split"] = cohort["scale_id"].map(manifest.set_index("scale_id")["split"])
parts = {name: cohort.loc[cohort["model_split"].eq(name)].reset_index(drop=True)
         for name in ["train", "validation", "test"]}
fish_sets = {name: set(df["fish_key"]) for name, df in parts.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not fish_sets[a] & fish_sets[b], f"Fish leakage: {a}/{b}"
assert len(manifest) == len(cohort) and cohort["model_split"].isin(parts).all()
display(pd.crosstab(cohort["model_split"], cohort["target"]))
print("Fish counts:", {k: len(v) for k, v in fish_sets.items()})
if parts["train"][texture_cols].isna().all().any():
    raise ValueError("A texture column is empty in training; remove it explicitly and restart.")
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
texture_arrays = {"train": scaler.fit_transform(imputer.fit_transform(
    parts["train"][texture_cols])).astype(np.float32)}
for name in ["validation", "test"]:
    texture_arrays[name] = scaler.transform(imputer.transform(
        parts[name][texture_cols])).astype(np.float32)
joblib.dump({"columns": texture_cols, "imputer": imputer, "scaler": scaler},
            OUTPUT_DIR / "texture_preprocessing.joblib")

Candidate plan and resume guard

Plans and successful/failed trial logs are saved. Completed trials are reused
only for matching data, split, search and execution settings. Use a new output
directory for another experiment. An out-of-memory trial is recorded as failed,
not silently rerun at a different batch size.

In [ ]:
def candidate_plan(space, count, baseline):
    if SEARCH_MODE not in {"random", "grid"}:
        raise ValueError("SEARCH_MODE must be random or grid.")
    grid = list(ParameterGrid(space))
    if not grid or count < 1:
        raise ValueError("Search spaces must be non-empty and budgets positive.")
    if SEARCH_MODE == "grid":
        return grid
    sampled = list(ParameterSampler(space, n_iter=len(grid), random_state=SEED))
    candidates = [baseline] if baseline in grid else []
    for candidate in sampled:
        if candidate not in candidates:
            candidates.append(candidate)
        if len(candidates) >= min(count, len(grid)):
            break
    return candidates[:min(count, len(grid))]

simclr_candidates = candidate_plan(SIMCLR_SPACE, N_PRETRAIN_TRIALS,
    {"lr": 3e-4, "temperature": 0.2, "batch_size": 128, "epochs": 10})
classifier_candidates = candidate_plan(CLASSIFIER_SPACE, N_CLASSIFIER_TRIALS,
    {"lr": 1e-4, "batch_size": 64})
if CLASSIFIER_MAX_EPOCHS < 1 or EARLY_STOP_PATIENCE < 1 or EARLY_STOP_MIN_DELTA < 0:
    raise ValueError("Epoch cap/patience must be positive and min_delta non-negative.")
if THRESHOLD_OBJECTIVE not in {"pipeline_macro_f1", "quality_balanced_accuracy"}:
    raise ValueError("Unknown threshold objective.")
if not THRESHOLD_GRID or any(not 0 <= value <= 1 for value in THRESHOLD_GRID):
    raise ValueError("Threshold grid must contain probabilities in [0, 1].")
if MAX_READABLE_REJECTION is not None and not 0 <= MAX_READABLE_REJECTION <= 1:
    raise ValueError("Readable rejection constraint must be in [0, 1].")
for config in simclr_candidates:
    if config["lr"] <= 0 or config["temperature"] <= 0 or config["batch_size"] < 2 or config["epochs"] < 1:
        raise ValueError("Invalid SimCLR candidate.")
for config in classifier_candidates:
    if config["lr"] <= 0 or config["batch_size"] < 2:
        raise ValueError("Invalid classifier candidate.")
experiments = [
    ("simclr_resnet18", "resnet18", False),
    ("simclr_resnet18_texture", "resnet18", True),
    ("simclr_resnet50_texture", "resnet50", True),
    ("simclr_resnet50", "resnet50", False),
]
split_digest = hashlib.sha256(pd.util.hash_pandas_object(
    manifest.sort_values("scale_id"), index=False).values.tobytes()).hexdigest()
search_config = {"protocol": "two_stage_validation_search_v1", "seed": SEED,
    "cohort_sha256": digest, "split_sha256": split_digest,
    "simclr_candidates": simclr_candidates, "classifier_candidates": classifier_candidates,
    "max_classifier_epochs": CLASSIFIER_MAX_EPOCHS, "patience": EARLY_STOP_PATIENCE,
    "min_delta": EARLY_STOP_MIN_DELTA, "eval_batch_size": EVAL_BATCH_SIZE,
    "threshold_grid": THRESHOLD_GRID, "threshold_objective": THRESHOLD_OBJECTIVE,
    "max_readable_rejection": MAX_READABLE_REJECTION,
    "num_workers": NUM_WORKERS, "use_amp": AMP_ENABLED,
    "torch_version": str(torch.__version__), "class_names": CLASS_NAMES,
    "quality_class_names": QUALITY_NAMES, "uses_length_weight": False}
search_signature = hashlib.sha256(json.dumps(search_config, sort_keys=True).encode()).hexdigest()
search_config_path = OUTPUT_DIR / "search_config.json"
if search_config_path.exists():
    if json.loads(search_config_path.read_text()) != search_config:
        raise ValueError("Search settings/data changed. Choose a new OUTPUT_DIR.")
else:
    search_config_path.write_text(json.dumps(search_config, indent=2))
TRIAL_DIR = OUTPUT_DIR / "search_trials"
TRIAL_DIR.mkdir(exist_ok=True)
display(pd.DataFrame(simclr_candidates))
display(pd.DataFrame(classifier_candidates))
print("SimCLR runs:", 2 * len(simclr_candidates))
print("Maximum classifier trials:", 8 * len(simclr_candidates) * len(classifier_candidates))

def release_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)

def atomic_checkpoint(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)

In [ ]:
normalization = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
simclr_transform = T.Compose([
    T.Resize((224, 224)), T.RandomResizedCrop(224, scale=(0.65, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(brightness=0.25, contrast=0.25), T.RandomGrayscale(p=0.2),
    T.ToTensor(), normalization,
])
# Match texture to the same unaugmented image; keep this identical for all heads.
classifier_transform = T.Compose([T.Resize((224, 224)), T.ToTensor(), normalization])

class ScaleDataset(Dataset):
    def __init__(self, frame, transform, texture=None, contrastive=False):
        self.frame = frame.reset_index(drop=True)
        self.transform, self.texture, self.contrastive = transform, texture, contrastive

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        with Image.open(self.frame.iloc[idx]["path"]) as image:
            image = image.convert("RGB")
        if self.contrastive:
            return self.transform(image), self.transform(image)
        return (self.transform(image), torch.from_numpy(self.texture[idx]),
                int(self.frame.iloc[idx]["target"]))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=NUM_WORKERS,
                      pin_memory=device.type == "cuda",
                      generator=torch.Generator().manual_seed(SEED))

def make_backbone(name, pretrained=True):
    builders = {"resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
                "resnet50": (models.resnet50, models.ResNet50_Weights.DEFAULT)}
    builder, weights = builders[name]
    backbone = builder(weights=weights if pretrained else None)
    feature_dim = backbone.fc.in_features
    backbone.fc = nn.Identity()
    return backbone, feature_dim

def ntxent(z1, z2, temperature=0.2):
    n = len(z1)
    if n < 2:
        raise ValueError("SimCLR requires at least two images for negative pairs.")
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)
    logits = z @ z.T / temperature
    logits = logits.masked_fill(torch.eye(2*n, device=z.device, dtype=torch.bool), -torch.inf)
    positive_indices = (torch.arange(2*n, device=z.device) + n) % (2*n)
    return F.cross_entropy(logits, positive_indices)

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

class AgeClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, texture_dim=0, n_classes=4):
        super().__init__()
        self.backbone = backbone
        self.texture_branch = (nn.Sequential(nn.Linear(texture_dim, 32), nn.ReLU())
                               if texture_dim else None)
        self.head = nn.Sequential(nn.Linear(feature_dim + (32 if texture_dim else 0), 128),
                                  nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes))

    def forward(self, images, texture):
        image_features = self.backbone(images)
        if self.texture_branch is not None:
            image_features = torch.cat([image_features, self.texture_branch(texture)], dim=1)
        return self.head(image_features)

Train two SimCLR backbones

Image-only and fusion start from the same SimCLR weights for each backbone.
SimCLR includes readable and bad training scales, with no held-out fish.
All classifier experiments fine-tune the full backbone, as in the old notebook.
Candidate lists are the same for both backbones. Record any infeasible trials
and compare completed counts as well as scores. AMP is fixed across all trials.

In [ ]:
def pretrain(name, config, trial_id):
    set_seed()
    checkpoint_path = TRIAL_DIR / f"{name}_{trial_id}_simclr.pt"
    status_path = TRIAL_DIR / f"{name}_{trial_id}_simclr_status.json"
    if RESUME_SEARCH and status_path.exists():
        status = json.loads(status_path.read_text())
        if status["search_signature"] != search_signature or status["config"] != config:
            raise ValueError("Stale pretraining trial.")
        if status["status"] != "completed":
            return None
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Completed trial checkpoint missing: {checkpoint_path}")
        return checkpoint_path
    started = time.perf_counter()
    backbone, dim = make_backbone(name)
    backbone = backbone.to(device)
    projector = nn.Sequential(nn.Linear(dim, 128), nn.ReLU(), nn.Linear(128, 128)).to(device)
    optimizer = torch.optim.AdamW(list(backbone.parameters()) + list(projector.parameters()),
                                  lr=config["lr"], weight_decay=1e-4)
    amp_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    ds = ScaleDataset(parts["train"], simclr_transform, contrastive=True)
    if len(ds) < config["batch_size"]:
        raise ValueError("Reduce the common SimCLR batch size; no full batch available.")
    loader = make_loader(ds, config["batch_size"], shuffle=True, drop_last=True)
    history = []
    for epoch in range(config["epochs"]):
        backbone.train(); projector.train()
        total, count = 0.0, 0
        for x1, x2 in tqdm(loader, desc=f"{name}/{trial_id} SimCLR {epoch+1}/{config['epochs']}"):
            x1, x2 = x1.to(device), x2.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                loss = ntxent(projector(backbone(x1)), projector(backbone(x2)), config["temperature"])
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite SimCLR loss.")
            amp_scaler.scale(loss).backward()
            amp_scaler.step(optimizer); amp_scaler.update()
            total += loss.item() * len(x1); count += len(x1)
        history.append({"epoch": epoch+1, "loss": total/count})
        print(history[-1])
    state = cpu_state(backbone)
    atomic_checkpoint(checkpoint_path, {"backbone_state_dict": state, "feature_dim": dim,
        "search_signature": search_signature, "config": config, "history": history})
    pd.DataFrame(history).to_csv(TRIAL_DIR / f"{name}_{trial_id}_pretraining_history.csv", index=False)
    atomic_json(status_path, {"status": "completed", "search_signature": search_signature,
        "config": config, "seconds": time.perf_counter()-started})
    return checkpoint_path

if RUN_TRAINING:
    pretrained_trials = {name: [] for name in ["resnet18", "resnet50"]}
    for backbone_name in pretrained_trials:
        for index, config in enumerate(simclr_candidates):
            trial_id = f"pre{index:03d}"
            try:
                path = pretrain(backbone_name, config, trial_id)
            except torch.cuda.OutOfMemoryError:
                release_gpu()
                atomic_json(TRIAL_DIR / f"{backbone_name}_{trial_id}_simclr_status.json",
                    {"status": "out_of_memory", "search_signature": search_signature, "config": config})
                print("Skipped OOM pretraining trial:", backbone_name, trial_id, config)
                path = None
            if path is not None:
                pretrained_trials[backbone_name].append({"trial_id": trial_id,
                                                        "config": config, "path": str(path)})
            release_gpu()
        if not pretrained_trials[backbone_name]:
            raise RuntimeError(f"No feasible SimCLR candidate for {backbone_name}; reduce search batch sizes.")
else:
    print("Audit the cohort above, then set RUN_TRAINING=True and run from the setup cell.")

In [ ]:
@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    truth, probabilities = [], []
    for images, texture, labels in loader:
        with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
            logits = model(images.to(device), texture.to(device))
        probabilities.append(logits.float().softmax(1).cpu().numpy())
        truth.extend(labels.numpy().tolist())
    probs = np.concatenate(probabilities)
    return np.asarray(truth), probs.argmax(1), probs

def metrics_for(truth, pred, names=CLASS_NAMES):
    report = classification_report(truth, pred, labels=list(range(len(names))), target_names=names,
                                   output_dict=True, zero_division=0)
    return {"accuracy": accuracy_score(truth, pred),
            "balanced_accuracy": balanced_accuracy_score(truth, pred),
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_f1": report["weighted avg"]["f1-score"],
            "support": len(truth)}, report

def task_data(task, splits=("train", "validation")):
    frames, arrays = {}, {}
    for split in splits:
        frame = parts[split]
        mask = frame["target"].lt(4).to_numpy() if task == "age" else np.ones(len(frame), dtype=bool)
        frames[split] = frame.loc[mask].copy().reset_index(drop=True)
        arrays[split] = texture_arrays[split][mask]
        if task == "quality":
            frames[split]["target"] = frames[split]["target"].eq(4).astype(int)
    return frames, arrays

def train_classifier(name, backbone_name, fusion, task, pretrial, config, trial_id):
    set_seed()
    started = time.perf_counter()
    names = CLASS_NAMES if task == "age" else QUALITY_NAMES
    labels = list(range(len(names)))
    task_parts, task_texture = task_data(task)
    backbone, dim = make_backbone(backbone_name, pretrained=False)
    pretrained = torch.load(pretrial["path"], map_location="cpu", weights_only=True)
    if pretrained["search_signature"] != search_signature:
        raise ValueError("Pretraining checkpoint belongs to another search.")
    backbone.load_state_dict(pretrained["backbone_state_dict"], strict=True)
    del pretrained
    model = AgeClassifier(backbone, dim, len(texture_cols) if fusion else 0, len(names)).to(device)
    loaders = {split: make_loader(ScaleDataset(frame, classifier_transform, task_texture[split]),
                                 config["batch_size"] if split == "train" else EVAL_BATCH_SIZE,
                                 shuffle=split == "train")
               for split, frame in task_parts.items()}
    counts = task_parts["train"]["target"].value_counts().reindex(labels, fill_value=0)
    if counts.eq(0).any():
        raise ValueError(f"Missing {task} class in training.")
    weights = torch.tensor((counts.sum()/(len(names)*counts)).to_numpy(), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
    amp_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    best_score, best_state, best_epoch = -1.0, None, None
    patience_score, stale_epochs = -1.0, 0
    history = []
    for epoch in range(CLASSIFIER_MAX_EPOCHS):
        model.train()
        total, count = 0.0, 0
        for images, texture, y in tqdm(loaders["train"], desc=f"{name}/{trial_id} {epoch+1}/{CLASSIFIER_MAX_EPOCHS}"):
            images, texture, y = images.to(device), texture.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                loss = criterion(model(images, texture), y)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite classifier loss.")
            amp_scaler.scale(loss).backward()
            amp_scaler.step(optimizer); amp_scaler.update()
            total += loss.item()*len(y); count += len(y)
        truth, pred, _ = predict_model(model, loaders["validation"])
        val_metrics, _ = metrics_for(truth, pred, names)
        history.append({"epoch": epoch+1, "train_loss": total/count,
                        **{f"val_{k}": v for k,v in val_metrics.items()}})
        print(history[-1])
        if val_metrics["macro_f1"] > best_score:
            best_score, best_epoch = val_metrics["macro_f1"], epoch+1
            best_state = cpu_state(model)
        if val_metrics["macro_f1"] > patience_score + EARLY_STOP_MIN_DELTA:
            patience_score, stale_epochs = val_metrics["macro_f1"], 0
        else:
            stale_epochs += 1
        if stale_epochs >= EARLY_STOP_PATIENCE:
            print("Early stopping:", name, trial_id, "best epoch:", best_epoch)
            break
    model.load_state_dict(best_state, strict=True)
    candidate_path = TRIAL_DIR / f"{name}_candidate.pt"
    atomic_checkpoint(candidate_path, {"model_state_dict": best_state, "backbone": backbone_name,
                "feature_dim": dim, "texture_cols": texture_cols if fusion else [],
                "class_names": names, "task": task, "best_epoch": best_epoch,
                "validation_macro_f1": best_score, "search_signature": search_signature,
                "cohort_sha256": digest, "classifier_config": config,
                "pretraining_config": pretrial["config"], "trial_id": trial_id})
    pd.DataFrame(history).to_csv(TRIAL_DIR / f"{name}_{trial_id}_history.csv", index=False)
    truth, pred, probs = predict_model(model, loaders["validation"])
    metrics, report = metrics_for(truth, pred, names)
    del model
    release_gpu()
    return {"model": name, "task": task, "status": "completed", "trial_id": trial_id,
            "best_epoch": best_epoch, "epochs_run": len(history),
            "validation_macro_f1": best_score, "validation_accuracy": metrics["accuracy"],
            "validation_balanced_accuracy": metrics["balanced_accuracy"],
            "pretraining_config": pretrial["config"], "classifier_config": config,
            "seconds": time.perf_counter()-started}

def search_classifier(name, backbone_name, fusion, task):
    log_path = TRIAL_DIR / f"{name}_trials.json"
    records = json.loads(log_path.read_text()) if RESUME_SEARCH and log_path.exists() else []
    completed_ids = {row["trial_id"] for row in records}
    best_path = OUTPUT_DIR / f"{name}_best.pt"
    if records and not best_path.exists() and any(row["status"] == "completed" for row in records):
        raise FileNotFoundError(f"Search log exists but selected checkpoint is missing: {best_path}")
    best_score = -1.0
    if RESUME_SEARCH and best_path.exists():
        checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
        if checkpoint["search_signature"] != search_signature:
            raise ValueError("Stale selected checkpoint.")
        best_score = checkpoint["validation_macro_f1"]
        del checkpoint
    for pretrial in pretrained_trials[backbone_name]:
        for index, config in enumerate(classifier_candidates):
            trial_id = f"{pretrial['trial_id']}_cls{index:03d}"
            if trial_id in completed_ids:
                continue
            try:
                row = train_classifier(name, backbone_name, fusion, task, pretrial, config, trial_id)
            except torch.cuda.OutOfMemoryError:
                release_gpu()
                row = {"model": name, "task": task, "trial_id": trial_id, "status": "out_of_memory",
                       "pretraining_config": pretrial["config"], "classifier_config": config}
                print("Skipped OOM classifier trial:", name, trial_id)
            if row["status"] == "completed" and row["validation_macro_f1"] > best_score:
                temporary = best_path.with_suffix(".pt.tmp")
                shutil.copyfile(TRIAL_DIR / f"{name}_candidate.pt", temporary)
                temporary.replace(best_path)
                best_score = row["validation_macro_f1"]
            records.append(row)
            atomic_json(log_path, records)
    completed = [row for row in records if row["status"] == "completed"]
    if not completed:
        raise RuntimeError(f"No feasible classifier candidate for {name}.")
    winner = max(completed, key=lambda row: row["validation_macro_f1"])
    pd.DataFrame(records).to_csv(TRIAL_DIR / f"{name}_search_history.csv", index=False)
    return {**winner, "completed_trials": len(completed), "failed_trials": len(records)-len(completed)}

Validation-only model search

Adding bad changes the fish cohort and split relative to the previous notebook.
Old checkpoints cannot be reused: a former training fish might now be in test.
Choose each age/quality model separately by validation macro F1. The same
candidate lists are applied to all four configurations. Test is not read here.
Fusion has a small extra texture branch, so gains may also reflect capacity.
Test predictions are retained for fish-level uncertainty analysis later.

In [ ]:
if RUN_TRAINING:
    selected_age, selected_quality = [], []
    for name, backbone_name, fusion in experiments:
        selected_age.append(search_classifier(name, backbone_name, fusion, "age"))
        selected_quality.append(search_classifier(name + "_quality", backbone_name, fusion, "quality"))
        pd.DataFrame(selected_age).to_csv(OUTPUT_DIR / "selected_age_validation_results.csv", index=False)
        pd.DataFrame(selected_quality).to_csv(OUTPUT_DIR / "selected_quality_validation_results.csv", index=False)
    display(pd.DataFrame(selected_age))
    display(pd.DataFrame(selected_quality))
else:
    print("Training disabled. Enable RUN_TRAINING in the setup cell when ready.")

Validation threshold search and frozen test evaluation

Evaluate each matched quality/age model pair on ALL labeled test scales.
A bad prediction stops the pipeline. Only scales passed by the gate are sent
to the age model. The five final outcomes are 0, 1, 2, 3 or older, and bad.
The age-only test table uses GT-readable scales (an oracle gate); it does not
measure deployment performance. The end-to-end table counts rejected readable
scales and passed bad scales as errors. Report age accuracy among accepted
GT-readable scales with coverage, never alone. Sweep the selected model pair
on validation to choose its bad threshold. Ties prefer a lower bad pass rate,
then a lower readable rejection rate, then proximity to 0.5.
Set EVALUATE_TEST=True only after the candidate spaces, models and thresholds
are frozen. Test results must not guide further tuning.

In [ ]:
def load_task_model(name, backbone_name, fusion, n_classes):
    checkpoint = torch.load(OUTPUT_DIR / f"{name}_best.pt", map_location="cpu", weights_only=True)
    if checkpoint["search_signature"] != search_signature:
        raise ValueError("Checkpoint belongs to another search/cohort/split.")
    backbone, dim = make_backbone(backbone_name, pretrained=False)
    model = AgeClassifier(backbone, dim, len(texture_cols) if fusion else 0, n_classes)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to(device)

def pipeline_statistics(truth, final_pred):
    readable = truth < 4
    bad = ~readable
    accepted = final_pred < 4
    accepted_readable = readable & accepted
    metrics, _ = metrics_for(truth, final_pred, PIPELINE_NAMES)
    return {**metrics,
            "readable_count": int(readable.sum()), "bad_count": int(bad.sum()),
            "readable_rejection_rate": float((~accepted[readable]).mean()),
            "bad_pass_rate": float(accepted[bad].mean()),
            "readable_coverage": float(accepted[readable].mean()),
            "readable_end_to_end_accuracy": float((final_pred[readable] == truth[readable]).mean()),
            "accepted_readable_age_accuracy": (float((final_pred[accepted_readable] == truth[accepted_readable]).mean())
                                                if accepted_readable.any() else np.nan)}

def validation_predictions(name, backbone_name, fusion):
    frame = parts["validation"]
    texture = texture_arrays["validation"]
    quality_frame = frame.copy()
    quality_frame["target"] = quality_frame["target"].eq(4).astype(int)
    model = load_task_model(name + "_quality", backbone_name, fusion, 2)
    loader = make_loader(ScaleDataset(quality_frame, classifier_transform, texture), EVAL_BATCH_SIZE)
    _, _, quality_probs = predict_model(model, loader)
    del model
    release_gpu()
    model = load_task_model(name, backbone_name, fusion, 4)
    loader = make_loader(ScaleDataset(frame, classifier_transform, texture), EVAL_BATCH_SIZE)
    _, age_pred, _ = predict_model(model, loader)
    del model
    release_gpu()
    return frame["target"].to_numpy(), quality_probs[:, 1], age_pred

def select_threshold(name, backbone_name, fusion):
    truth, bad_prob, age_pred = validation_predictions(name, backbone_name, fusion)
    rows = []
    for threshold in THRESHOLD_GRID:
        rejected = bad_prob >= threshold
        final_pred = np.where(rejected, 4, age_pred)
        stats = pipeline_statistics(truth, final_pred)
        rows.append({"threshold": float(threshold),
                     "pipeline_macro_f1": stats["macro_f1"],
                     "quality_balanced_accuracy": float(balanced_accuracy_score(truth == 4, rejected)),
                     "readable_rejection_rate": stats["readable_rejection_rate"],
                     "bad_pass_rate": stats["bad_pass_rate"],
                     "readable_coverage": stats["readable_coverage"]})
    curve = pd.DataFrame(rows)
    curve.to_csv(OUTPUT_DIR / f"{name}_validation_threshold_curve.csv", index=False)
    feasible = curve if MAX_READABLE_REJECTION is None else curve.loc[
        curve["readable_rejection_rate"] <= MAX_READABLE_REJECTION]
    if feasible.empty:
        raise ValueError(f"No feasible threshold for {name}; reconsider the validation constraint.")
    feasible = feasible.assign(distance_to_half=(feasible["threshold"] - 0.5).abs())
    winner = feasible.sort_values(
        [THRESHOLD_OBJECTIVE, "bad_pass_rate", "readable_rejection_rate", "distance_to_half", "threshold"],
        ascending=[False, True, True, True, True]).iloc[0]
    return {"model": name, "objective": THRESHOLD_OBJECTIVE,
            **{column: float(winner[column]) for column in curve.columns}}

THRESHOLD_PATH = OUTPUT_DIR / "selected_thresholds.json"
if RUN_TRAINING:
    selected_thresholds = [select_threshold(*experiment) for experiment in experiments]
    atomic_json(THRESHOLD_PATH, {"search_signature": search_signature, "models": selected_thresholds})
    display(pd.DataFrame(selected_thresholds))
    print("Selection complete. Freeze the search before enabling EVALUATE_TEST.")
else:
    print("Validation threshold selection runs after model search.")

Search diagnostics

Inspect validation scores, best epochs, actual training duration and failed
trials before testing. Curves show the cost of rejecting readable scales
versus passing bad scales. These are validation diagnostics, not test results.

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt
    trial_tables = []
    for name, _, _ in experiments:
        for suffix in ["", "_quality"]:
            trial_tables.extend(json.loads((TRIAL_DIR / f"{name}{suffix}_trials.json").read_text()))
    search_history_df = pd.DataFrame(trial_tables)
    display(search_history_df)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for axis, (name, _, _) in zip(axes.flat, experiments):
        curve = pd.read_csv(OUTPUT_DIR / f"{name}_validation_threshold_curve.csv")
        for metric in [THRESHOLD_OBJECTIVE, "readable_rejection_rate", "bad_pass_rate"]:
            axis.plot(curve["threshold"], curve[metric], label=metric)
        selected = next(row for row in selected_thresholds if row["model"] == name)
        axis.axvline(selected["threshold"], color="black", linestyle="--")
        axis.set(title=name, xlabel="Bad probability threshold", ylabel="Validation score / rate", ylim=(0, 1))
        axis.legend(fontsize=8)
    plt.show()

In [ ]:
def save_report(name, frame, truth, pred, names, probs):
    metrics, report = metrics_for(truth, pred, names)
    labels = list(range(len(names)))
    pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f"{name}_classification_report.csv")
    cm = pd.DataFrame(confusion_matrix(truth, pred, labels=labels), index=names, columns=names)
    cm.to_csv(OUTPUT_DIR / f"{name}_confusion_matrix.csv")
    predictions = frame[["scale_id", "fish_key", "target"]].copy()
    predictions["prediction"] = pred
    for index in labels:
        predictions[f"prob_{index}"] = probs[:, index]
    predictions.to_csv(OUTPUT_DIR / f"{name}_predictions.csv", index=False)
    print(name)
    print(classification_report(truth, pred, labels=labels, target_names=names, digits=4, zero_division=0))
    display(cm)
    return metrics

def evaluate_task(name, backbone_name, fusion, task, threshold):
    frames, arrays = task_data(task, splits=("test",))
    names = CLASS_NAMES if task == "age" else QUALITY_NAMES
    frame = frames["test"]
    model = load_task_model(name, backbone_name, fusion, len(names))
    loader = make_loader(ScaleDataset(frame, classifier_transform, arrays["test"]), EVAL_BATCH_SIZE)
    truth, pred, probs = predict_model(model, loader)
    if task == "quality":
        pred = (probs[:, 1] >= threshold).astype(int)
    del model
    release_gpu()
    metrics = save_report(name, frame, truth, pred, names, probs)
    checkpoint = torch.load(OUTPUT_DIR / f"{name}_best.pt", map_location="cpu", weights_only=True)
    return {"model": name, **metrics, "best_epoch": checkpoint["best_epoch"],
            "validation_macro_f1": checkpoint["validation_macro_f1"],
            **({"bad_threshold": threshold} if task == "quality" else {})}

def evaluate_pipeline(name, backbone_name, fusion, threshold):
    test = parts["test"]
    texture = texture_arrays["test"]
    quality_model = load_task_model(name + "_quality", backbone_name, fusion, 2)
    quality_frame = test.copy()
    quality_frame["target"] = quality_frame["target"].eq(4).astype(int)
    loader = make_loader(ScaleDataset(quality_frame, classifier_transform, texture), EVAL_BATCH_SIZE)
    _, _, quality_probs = predict_model(quality_model, loader)
    passed = quality_probs[:, 1] < threshold
    final_pred = np.full(len(test), 4, dtype=int)
    age_probs = np.full((len(test), 4), np.nan)
    del quality_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if passed.any():
        age_model = load_task_model(name, backbone_name, fusion, 4)
        age_frame = test.loc[passed].reset_index(drop=True)
        loader = make_loader(ScaleDataset(age_frame, classifier_transform, texture[passed]), EVAL_BATCH_SIZE)
        _, age_pred, probs = predict_model(age_model, loader)
        final_pred[passed] = age_pred
        age_probs[passed] = probs
        del age_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    truth = test["target"].to_numpy()
    row = {"model": name, "bad_threshold": threshold, **pipeline_statistics(truth, final_pred)}
    predictions = test[["scale_id", "fish_key", "target"]].copy()
    predictions["bad_probability"] = quality_probs[:, 1]
    predictions["gate_passed"] = passed
    predictions["final_prediction"] = final_pred
    for k in LABELS:
        predictions[f"age_prob_{k}"] = age_probs[:, k]
    predictions.to_csv(OUTPUT_DIR / f"{name}_pipeline_predictions.csv", index=False)
    cm = pd.DataFrame(confusion_matrix(truth, final_pred, labels=SPLIT_LABELS),
                      index=PIPELINE_NAMES, columns=PIPELINE_NAMES)
    cm.to_csv(OUTPUT_DIR / f"{name}_pipeline_confusion_matrix.csv")
    _, report = metrics_for(truth, final_pred, PIPELINE_NAMES)
    pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f"{name}_pipeline_classification_report.csv")
    print(name)
    print(classification_report(truth, final_pred, labels=SPLIT_LABELS,
                                target_names=PIPELINE_NAMES, digits=4, zero_division=0))
    display(cm)
    return row

if EVALUATE_TEST:
    frozen_selection = json.loads(THRESHOLD_PATH.read_text())
    if frozen_selection["search_signature"] != search_signature:
        raise ValueError("Thresholds belong to another search protocol.")
    thresholds = {row["model"]: row["threshold"] for row in frozen_selection["models"]}
    results, quality_results, pipeline_results = [], [], []
    for name, backbone_name, fusion in experiments:
        threshold = thresholds[name]
        results.append(evaluate_task(name, backbone_name, fusion, "age", threshold))
        quality_results.append(evaluate_task(name + "_quality", backbone_name, fusion, "quality", threshold))
        pipeline_results.append(evaluate_pipeline(name, backbone_name, fusion, threshold))
    results_df, quality_results_df, pipeline_results_df = map(pd.DataFrame, [results, quality_results, pipeline_results])
    results_df.to_csv(OUTPUT_DIR / "comparison_results.csv", index=False)
    quality_results_df.to_csv(OUTPUT_DIR / "quality_comparison_results.csv", index=False)
    pipeline_results_df.to_csv(OUTPUT_DIR / "pipeline_comparison_results.csv", index=False)
    display(results_df)
    display(quality_results_df)
    display(pipeline_results_df)
else:
    print("Test evaluation disabled. Set EVALUATE_TEST=True only after freezing selection.")